# Keeping the extremes

Suppose a model sizes power plants for the highest load and must-run units for the lowest.
It needs the real extremes. `Distribution()` fits the duration curve well, but each of its
values is an average of several hours, the highest and the lowest one included.
`preserve_minmax=True` keeps the real extremes instead.

In this tutorial we find the extremes of two weeks of electricity load, see what
`Distribution()` makes of them, keep them with `preserve_minmax=True`, and count how often
they then occur. By the end you will have seen that a kept extreme occurs once for every day
its typical day represents.

This is part 3 of five [duration-curve tutorials](../index.md#duration-curves). Each part stands on its own.

## 1  Two weeks, two typical days

We take **1–14 April 2010** from the example data: hourly solar irradiance (`GHI`) and
electricity `Load`. Two typical days for two weeks is a deliberately tight budget; it makes
the differences easy to see.

Every run in this tutorial uses the same settings, so we write them down once in a small
function. Only its `representation` argument will change. `preserve_column_means=False`
switches off tsam's rescaling step, so that we see the values exactly as the representation
produced them.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

import tsam
from tsam import ClusterConfig, Distribution
from tsam.metrics import series_statistics
from tsam.plot import compare_series

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-04-01":"2010-04-14", ["GHI", "Load"]]
UNITS = {"GHI": "W/m²", "Load": "MW"}
COLORS = {
    "original": "#000000",
    "distribution": "#009E73",
    "preserve_minmax": "#8759A5",
}


def aggregate(representation):
    return tsam.aggregate(
        data,
        n_clusters=2,
        period_duration="1D",
        cluster=ClusterConfig(method="hierarchical", representation=representation),
        preserve_column_means=False,
    )

## 2  The extremes of the two weeks

Find the highest and the lowest load:

In [ ]:
peak, low = data["Load"].max(), data["Load"].min()
pd.DataFrame(
    {
        "when": [data["Load"].idxmax(), data["Load"].idxmin()],
        "Load (MW)": [peak, low],
    },
    index=["highest", "lowest"],
).round(1)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert int(np.isclose(data["Load"], peak).sum()) == 1
assert int(np.isclose(data["Load"], low).sum()) == 1

The highest load, 562.5 MW, occurs in one hour, on 9 April at 12:30. The lowest, 270 MW, also
occurs in one hour, on 2 April at 20:30.

## 3  What `Distribution()` makes of them

`result.reconstructed` lays the typical days back out over the two weeks, one copy for every
day each of them represents. Compare its load with the original:

In [ ]:
distribution = aggregate(Distribution())

series_statistics(
    {"original": data, "distribution": distribution.reconstructed}, columns=["Load"]
).round(1)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert round(peak - distribution.reconstructed["Load"].max()) == 5
assert round(distribution.reconstructed["Load"].min() - low) == 70

The highest load comes out 5 MW too low, the lowest 70 MW too high. Each end of a fitted
duration curve is an average. The highest value of the typical day for 9 days, for example,
is the average of the 9 highest hours of those days.

## 4  Keep the extremes

Change only one argument:

In [ ]:
minmax = aggregate(Distribution(preserve_minmax=True))

series_statistics(
    {
        "original": data,
        "distribution": distribution.reconstructed,
        "preserve_minmax": minmax.reconstructed,
    },
    columns=["Load"],
).round(1)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert np.isclose(minmax.reconstructed["Load"].max(), peak)
assert np.isclose(minmax.reconstructed["Load"].min(), low)
np.testing.assert_allclose(minmax.reconstructed["Load"].mean(), data["Load"].mean())

Now both extremes are exact, and the mean hasn't changed: tsam adjusts the values in between
to keep the mean of each typical day.

## 5  How often do they occur?

Zoom into the highest 5% of the load's duration curve:

In [ ]:
three = {
    "original": data,
    "distribution": distribution.reconstructed,
    "preserve_minmax": minmax.reconstructed,
}
compare_series(
    three,
    mode="duration_curve",
    duration_range=(0, 5),
    columns=["Load"],
    reference="original",
    units=UNITS,
    colors=COLORS,
    title="The highest 5% of the load",
).show()

The original reaches 562.5 MW for a single hour. With `preserve_minmax`, the curve stays at
562.5 MW for longer. Now the lowest 5%:

In [ ]:
compare_series(
    three,
    mode="duration_curve",
    duration_range=(95, 100),
    columns=["Load"],
    reference="original",
    units=UNITS,
    colors=COLORS,
    title="The lowest 5% of the load",
).show()

The same happens at the bottom: with `preserve_minmax`, the curve stays at 270 MW for longer.
Count the hours:

In [ ]:
pd.DataFrame(
    {
        name: {
            "hours at the highest load": int(np.isclose(frame["Load"], peak).sum()),
            "hours at the lowest load": int(np.isclose(frame["Load"], low).sum()),
        }
        for name, frame in three.items()
    }
).T

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert int(np.isclose(minmax.reconstructed["Load"], peak).sum()) == 9
assert int(np.isclose(minmax.reconstructed["Load"], low).sum()) == 9

Both extremes now last 9 hours instead of one. They both fall on days that belong to the
larger typical day, which represents 9 days. A kept extreme is part of that typical day, so it
occurs every time the typical day does.

## 6  What it costs

The extremes are exact now, but compare the duration-curve error, `rmse_duration`, in
normalized units (lower is better):

In [ ]:
pd.DataFrame(
    {
        "distribution": distribution.accuracy.rmse_duration,
        "preserve_minmax": minmax.accuracy.rmse_duration,
    }
).round(4)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert (
    minmax.accuracy.rmse_duration["Load"]
    > 2 * distribution.accuracy.rmse_duration["Load"]
)

The load's error more than doubles: two single hours have become 9 hours each. Solar hardly
changes, because `Distribution()` already came close to its extremes: 0 W/m² at night, and
a highest value within 10 W/m² of the real one.

## What you've learned

* Each end of a fitted duration curve is an average, so `Distribution()` misses the real
  extremes: here by 5 MW at the top and 70 MW at the bottom.
* `preserve_minmax=True` keeps each typical day's real lowest and highest values, and its mean.
* A kept extreme occurs once for every day its typical day represents: one hour of the
  original became 9.
* Use it when the extreme value itself matters to your model, and check what the repetition
  does to the rest of the results.

Where to go next:

* [Part 1: Fitting a duration curve](01_fitting.ipynb) — how the fitted values are built.
* [Part 2: Local or global](02_scope.ipynb) — fit each typical day's own duration curve, or
  the whole series'.
* [Part 4: Which values occur together](04_ordering.ipynb) — place the fitted values in time.
* [Part 5: Making peaks coincide on purpose](05_coincident_peaks.ipynb) — when pairing the
  highest values with each other is exactly what you want.